# GeoAI Aquaculture — Colab pull-run loop

**One-time setup:**
1. Drive: put `Train.csv`, `Test.csv`, `SampleSubmission.csv` in `MyDrive/geoai-data/`.
2. GitHub fine-grained PAT (repo `geoai-aquaculture`, Contents: Read-only) → add as Colab Secret `GH_PAT` (🔑 sidebar, enable notebook access).
3. **Runtime ▸ Change runtime type ▸ T4 GPU.**

**Each iteration:** Claude pushes an experiment → you **Runtime ▸ Run all** → upload the downloaded `submission_*.csv` on Zindi → paste the LB score back.

The experiment itself lives in `experiments/run_current.sh` (version-controlled), so this notebook rarely changes.

In [ ]:
# Cell 1 — pull latest code (private repo via PAT secret)
import os
from google.colab import userdata
PAT  = userdata.get('GH_PAT')
REPO = 'OsbornNyakaru/geoai-aquaculture'
if not os.path.exists('/content/geoai/.git'):
    !git clone https://{PAT}@github.com/{REPO}.git /content/geoai
%cd /content/geoai
!git pull --ff-only
!git log --oneline -1        # confirms which experiment you're running

In [ ]:
# Cell 2 — GPU check + the two libs Colab lacks (torch/xgboost/sklearn preinstalled)
!nvidia-smi -L || echo 'NO GPU — set Runtime ▸ Change runtime type ▸ T4 GPU'
!pip -q install lightgbm==4.6.0 catboost==1.2.10 pyyaml
import torch; print('CUDA available:', torch.cuda.is_available())

In [ ]:
# Cell 3 — data from Drive (one-time folder MyDrive/geoai-data/)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/geoai/data/raw
!cp /content/drive/MyDrive/geoai-data/*.csv /content/geoai/data/raw/
!ls -1 /content/geoai/data/raw          # expect Train.csv Test.csv SampleSubmission.csv

In [ ]:
# Cell 4 — run THIS iteration's experiment (Claude sets the commands in run_current.sh)
%cd /content/geoai
!bash experiments/run_current.sh

In [ ]:
# Cell 5 — download the newest submission(s) to upload on Zindi
from google.colab import files
import glob, os
for f in sorted(glob.glob('submissions/submission_*.csv'), key=os.path.getmtime)[-4:]:
    print('↓', f); files.download(f)

In [ ]:
# Cell 5b — ship the DIAGNOSTIC BUNDLES home (added iter44).
#
# WHY THIS EXISTS. run_pipeline.py has always written submissions/preds/preds_<name>.npz --
# oof_prob, y, p_test_raw, test_per_fold, and (since iter44) the per-view OOF record. But Cell 5
# only ever downloaded the CSVs and submissions/preds/ is gitignored, so every bundle died with
# the Colab VM. That is the single reason the binormal b, the F1-optimal cut F*/2, and the test
# positive count P have all been ARGUED from leaderboard arithmetic instead of MEASURED on
# labelled data. One missing copy, forty-odd iterations.
#
# These are small (well under a megabyte each) and they are what tools/regime_match.py consumes.
import glob, os
n = len(glob.glob('submissions/preds/*.npz'))
print(f'{n} preds bundle(s) to copy')

# Primary: Drive (durable -- survives a lost runtime, and Cell 3 already mounted it).
!mkdir -p /content/drive/MyDrive/geoai-preds
!cp submissions/preds/*.npz /content/drive/MyDrive/geoai-preds/ && echo 'copied to MyDrive/geoai-preds/'
!ls -1 /content/drive/MyDrive/geoai-preds | tail -20

# Fallback: a single zip download, in case Drive sync lags or the mount dropped.
!cd submissions && zip -qr /content/preds.zip preds
from google.colab import files
files.download('/content/preds.zip')

# ⚠️ These bundles contain `y` (train labels) and `test_ids`, both derived from the competition
# CSVs. submissions/preds/ is gitignored -- keep it that way. Unzip locally INTO submissions/preds/
# and never `git add -f` them.

In [ ]:
# Cell 6 (optional) — reproducibility proof for the Phase-2 rubric
# Re-run Cell 4 once more; the two `final_oof` values must be byte-identical.